# 04 — Feature Engineering
**Movie Review Sentiment Analysis Project**

Stage 6: Feature Engineering — converting text to numerical features.

We explain and demonstrate:
- **Bag of Words (BoW)**: word count representation
- **TF-IDF**: Term Frequency–Inverse Document Frequency

> **Important:** Vectorizers are fitted on training data ONLY.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)
from src.data_loader import load_imdb_dataset, get_train_test
from src.preprocessing import apply_cleaning

df = load_imdb_dataset(PROJECT_ROOT)
train_df, test_df = get_train_test(df)
train_df['clean_review'] = apply_cleaning(train_df['review'])
test_df['clean_review'] = apply_cleaning(test_df['review'])

X_train_full = train_df['clean_review'].values
y_train_full = train_df['label'].values
X_test = test_df['clean_review'].values
y_test = test_df['label'].values

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)
print('Train:', len(X_train), '| Val:', len(X_val), '| Test:', len(X_test))

## A. Bag of Words (BoW)
Each review is represented by the count of words in a fixed vocabulary.  
Result: a sparse matrix of shape (num_reviews × vocabulary_size).

In [ ]:
bow = CountVectorizer(max_features=50_000, min_df=2)
X_train_bow = bow.fit_transform(X_train)  # fit ONLY on training data
X_val_bow   = bow.transform(X_val)         # transform without fitting

print(f'BoW matrix shape (train): {X_train_bow.shape}')
print(f'Vocabulary size: {len(bow.vocabulary_):,}')
print(f'Matrix density: {X_train_bow.nnz / (X_train_bow.shape[0] * X_train_bow.shape[1]):.4%}')
print(f'Sample vocab (first 10): {list(bow.vocabulary_.keys())[:10]}')

## B. TF-IDF (Term Frequency–Inverse Document Frequency)
- **TF** (Term Frequency): how often a word appears in a review.
- **IDF** (Inverse Document Frequency): penalises words common across many reviews.
- Result: words that are distinctive for a review get higher weight.

We use **unigrams + bigrams** to capture phrases like `not good`, `very bad`.

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),   # unigrams and bigrams
    sublinear_tf=True,    # log(1 + tf) smoothing
    min_df=2,             # ignore terms appearing in < 2 documents
    max_features=100_000,
)
X_train_tfidf = tfidf.fit_transform(X_train)  # fit ONLY on training data
X_val_tfidf   = tfidf.transform(X_val)

print(f'TF-IDF matrix shape (train): {X_train_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_):,}')
print(f'Sample vocab (first 10): {list(tfidf.vocabulary_.keys())[:10]}')

## C. Word2Vec / Embeddings (Explained)
Word2Vec learns dense vector representations of words from their context.  
Unlike BoW/TF-IDF, it captures semantic similarity (e.g. "good" ≈ "great").

**For this project** we use TF-IDF, which is appropriate for a classical NLP/ML baseline with scikit-learn.  
Word2Vec is documented here as an alternative approach (not implemented).

If Word2Vec were used, the process would be:
1. Train `gensim.models.Word2Vec` on training reviews.
2. Average word vectors per review to get a fixed-size feature vector.
3. Train a classifier on these averaged vectors.

Transformers (e.g. BERT) are a more powerful modern alternative but require more resources.